In [0]:
%run ./config

In [0]:
import boto3
import json
import uuid
import random
import time
from datetime import datetime,timedelta
from faker import Faker

In [0]:
#Connecting to AWS S3 

s3=boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY
)

In [0]:
#Generating random Events 

fake=Faker()
STATUSES=["placed","confirmed","shipped","delivered","cancelled"]
PRODUCTS=["Laptop","Phone","Headphones","Shoes","Watch","Backpack","Monitor","Keyboard"]

def generate_order():
    num_items=random.randint(1,4)
    
    #generates num_items list of items 
    items=[
        {
            "product":random.choice(PRODUCTS),
            "quantity":random.randint(1,3),
            "unit_price":round(random.uniform(10,500),2)
        }
        for _ in range(num_items)
    ]
    amount=round(sum(i['quantity']*i['unit_price'] for i in items),2)

    #converting UTC to IST
    IST=datetime.utcnow()+timedelta(hours=5,minutes=30)

    return {
        "order_id":str(uuid.uuid4()),
        "customer_id":fake.uuid4(),
        "items":items,
        "amount":amount,
        "status":random.choice(STATUSES),
        "order_timestamp":IST.isoformat()
    }

def generate_bad_order():
    #intentionally produce bad order
    bad_type=random.choice(["negative_amount","null_order_id","invalid_status"])
    order=generate_order()
    if bad_type=="negative_amount":
        order['amount']=round(random.uniform(-500,-1),2)
    elif bad_type=="null_order_id":
        order['order_id']=None
    elif bad_type=="invalid_status":
        order['status']="unknown_status"
    
    return order

In [0]:
#Writing to S3
def write_batch_to_s3(batch_size):
    now=datetime.utcnow()+timedelta(hours=5,minutes=30)

    #maximum 10% should be bad orders per batch
    num_bad=int(batch_size*random.uniform(0,0.1))
    num_good=batch_size-num_bad

    #Generating orders
    orders=[generate_order() for _ in range(num_good)]+[generate_bad_order() for _ in range(num_bad)] 

    #mixing orders rather than bad ones all at end
    random.shuffle(orders)

    #defining S3 path for json files
    key=f"raw/orders/{now.strftime('%Y/%m/%d')}/order_{now.strftime('%H:%M:%S')}_{uuid.uuid4().hex[:8]}.json"

    #appending each order of a batch in new line
    body="\n".join(json.dumps(o) for o in orders)
    
    #writing to the specified json file path
    s3.put_object(Bucket=BUCKET_NAME,Key=key,Body=body)

    print(f"Wrote {batch_size} orders ({num_bad} intentioanlly bad)->s3://{BUCKET_NAME}/{key}")

In [0]:
NUM_BATCHES=10 # no of batches
BATCH_INTERVAL=15 # wait time in sec per each batch

for i in range(NUM_BATCHES):
    batch_size=random.randint(1,50) #random batch size for one file
    write_batch_to_s3(batch_size)
    if i<NUM_BATCHES-1:time.sleep(BATCH_INTERVAL) #waiting  per batch

print("Done Simulating Stream.!")